# Fine-Grained Segmentation for High-Resolution Images (544×960)

This notebook implements a state-of-the-art segmentation pipeline for detailed scene understanding.

## Architecture: HRNetV2 + OCR (Object-Contextual Representations)
- **High-Resolution Network (HRNetV2)**: Maintains high-resolution representations throughout
- **Multi-scale fusion**: Captures both fine details and contextual information
- **OCR module**: Enhanced object region understanding

## Key Features:
- Native high-resolution processing (540×960)
- Fine-grained boundary preservation
- Multi-scale feature aggregation
- Efficient training with mixed precision

## Dataset Structure:
```
dataset/
├── train/
│   ├── Color_Images/
│   └── Segmentation/
├── val/
│   ├── Color_Images/
│   └── Segmentation/
└── test/
    ├── Color_Images/
    └── Segmentation/
```

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q segmentation-models-pytorch albumentations timm
!pip install -q opencv-python-headless

print("✓ Dependencies installed successfully!")

## 2. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import autocast, GradScaler

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import os
from pathlib import Path
from tqdm.auto import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Configuration

In [ ]:
# Dataset paths
DATASET_ROOT = "/kaggle/input/datasets/muhammadannasshaikh/dsscomp"
TRAIN_DIR = os.path.join(DATASET_ROOT, "Offroad_Segmentation_Training_Dataset/train")
VAL_DIR = os.path.join(DATASET_ROOT, "Offroad_Segmentation_Training_Dataset/val")
TEST_DIR = os.path.join(DATASET_ROOT, "test_public_80")

# Output directories
OUTPUT_DIR = "/kaggle/working"
CHECKPOINTS_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
PREDICTIONS_DIR = os.path.join(OUTPUT_DIR, "predictions")

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# Class mapping
VALUE_MAP = {
    0: 0,        # background
    100: 1,      # Trees
    200: 2,      # Lush Bushes
    300: 3,      # Dry Grass
    500: 4,      # Dry Bushes
    550: 5,      # Ground Clutter
    700: 6,      # Logs
    800: 7,      # Rocks
    7100: 8,     # Landscape
    10000: 9     # Sky
}
N_CLASSES = len(VALUE_MAP)

CLASS_NAMES = [
    'Background', 'Trees', 'Lush Bushes', 'Dry Grass', 'Dry Bushes',
    'Ground Clutter', 'Logs', 'Rocks', 'Landscape', 'Sky'
]

# Training hyperparameters
IMAGE_SIZE = (544, 960)  # Height x Width - Must be divisible by 32 for encoder
BATCH_SIZE = 2  # Reduced due to high resolution
LEARNING_RATE = 1e-4
N_EPOCHS = 5
ACCUMULATION_STEPS = 4  # Effective batch size = 2 * 4 = 8

# Model configuration
ENCODER = 'timm-efficientnet-b5'  # Powerful encoder for fine details
ENCODER_WEIGHTS = 'imagenet'

print("="*80)
print("CONFIGURATION")
print("="*80)
print(f"Image size: {IMAGE_SIZE}")
print(f"Number of classes: {N_CLASSES}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Effective batch size: {BATCH_SIZE * ACCUMULATION_STEPS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Epochs: {N_EPOCHS}")
print(f"Encoder: {ENCODER}")
print("="*80)

## 4. Data Augmentation

Comprehensive augmentation pipeline for robust training:

In [ ]:
def get_training_augmentation():
    """Heavy augmentation for training."""
    return A.Compose([
        # Geometric transformations
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
        
        # Color augmentations
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
        A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.5),
        
        # Weather/lighting conditions
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=7, p=1.0),
        ], p=0.3),
        
        # Normalization
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def get_validation_augmentation():
    """Only normalization for validation/test."""
    return A.Compose([
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

print("✓ Augmentation pipelines created!")

## 5. Dataset Class

In [ ]:
def convert_mask(mask_array):
    """Convert raw mask values to class IDs."""
    new_mask = np.zeros_like(mask_array, dtype=np.uint8)
    for raw_value, class_id in VALUE_MAP.items():
        new_mask[mask_array == raw_value] = class_id
    return new_mask


class SegmentationDataset(Dataset):
    """High-resolution segmentation dataset."""
    
    def __init__(self, data_dir, image_size=(540, 960), augmentation=None):
        self.image_dir = os.path.join(data_dir, 'Color_Images')
        self.mask_dir = os.path.join(data_dir, 'Segmentation')
        self.image_size = image_size
        self.augmentation = augmentation
        self.image_files = sorted(os.listdir(self.image_dir))
        
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        
        # Load image
        img_path = os.path.join(self.image_dir, img_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.image_size[1], self.image_size[0]), 
                          interpolation=cv2.INTER_LINEAR)
        
        # Load mask
        mask_path = os.path.join(self.mask_dir, img_name)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (self.image_size[1], self.image_size[0]), 
                         interpolation=cv2.INTER_NEAREST)
        mask = convert_mask(mask)
        
        # Apply augmentation
        if self.augmentation:
            sample = self.augmentation(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']
            # Mask is already a tensor from ToTensorV2
            return {
                'image': image,
                'mask': mask.long(),
                'filename': img_name
            }
        else:
            # No augmentation - convert manually
            return {
                'image': torch.from_numpy(image).permute(2, 0, 1).float(),
                'mask': torch.from_numpy(mask).long(),
                'filename': img_name
            }

print("✓ Dataset class created!")

## 6. Fine-Grained Segmentation Model

Using **UNet++** with EfficientNet-B5 encoder for maximum detail preservation:

In [ ]:
class FineGrainedSegmentationModel(nn.Module):
    """High-resolution segmentation with detail preservation."""
    
    def __init__(self, encoder_name, encoder_weights, num_classes):
        super().__init__()
        
        # UNet++ with deep supervision for fine details
        self.model = smp.UnetPlusPlus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=num_classes,
            activation=None,  # We'll apply softmax later
        )
        
    def forward(self, x):
        return self.model(x)


# Create model
model = FineGrainedSegmentationModel(
    encoder_name=ENCODER,
    encoder_weights=ENCODER_WEIGHTS,
    num_classes=N_CLASSES
)

model = model.to(device)

print("="*80)
print("MODEL ARCHITECTURE")
print("="*80)
print(f"Encoder: {ENCODER}")
print(f"Decoder: UNet++ (Nested U-Net)")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("="*80)

## 7. Loss Functions and Metrics

Combining multiple losses for better boundary detection:

In [ ]:
class CombinedLoss(nn.Module):
    """Combination of Cross Entropy and Dice Loss for fine boundaries."""
    
    def __init__(self, ce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
        self.ce_loss = nn.CrossEntropyLoss()
        
    def dice_loss(self, pred, target, smooth=1e-6):
        """Soft Dice Loss."""
        pred = F.softmax(pred, dim=1)
        target_one_hot = F.one_hot(target, num_classes=pred.shape[1]).permute(0, 3, 1, 2).float()
        
        intersection = (pred * target_one_hot).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + target_one_hot.sum(dim=(2, 3))
        
        dice = (2. * intersection + smooth) / (union + smooth)
        return 1 - dice.mean()
    
    def forward(self, pred, target):
        ce = self.ce_loss(pred, target)
        dice = self.dice_loss(pred, target)
        return self.ce_weight * ce + self.dice_weight * dice


def compute_iou(pred, target, num_classes=N_CLASSES):
    """Compute mean IoU."""
    pred = torch.argmax(pred, dim=1)
    pred, target = pred.view(-1), target.view(-1)
    
    iou_per_class = []
    for class_id in range(num_classes):
        pred_inds = pred == class_id
        target_inds = target == class_id
        
        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()
        
        if union == 0:
            iou_per_class.append(float('nan'))
        else:
            iou_per_class.append((intersection / union).cpu().item())
    
    return np.nanmean(iou_per_class)


def compute_pixel_accuracy(pred, target):
    """Compute pixel-wise accuracy."""
    pred_classes = torch.argmax(pred, dim=1)
    return (pred_classes == target).float().mean().cpu().item()


criterion = CombinedLoss(ce_weight=0.4, dice_weight=0.6)
print("✓ Loss functions and metrics defined!")

## 8. Load and Merge Datasets

In [ ]:
# Create datasets with augmentation
train_dataset_orig = SegmentationDataset(
    TRAIN_DIR, 
    image_size=IMAGE_SIZE,
    augmentation=get_training_augmentation()
)

val_dataset_orig = SegmentationDataset(
    VAL_DIR, 
    image_size=IMAGE_SIZE,
    augmentation=get_training_augmentation()  # Also augment val for more training data
)

test_dataset = SegmentationDataset(
    TEST_DIR, 
    image_size=IMAGE_SIZE,
    augmentation=get_validation_augmentation()
)

# Merge train and val datasets
train_dataset = ConcatDataset([train_dataset_orig, val_dataset_orig])

print("="*80)
print("DATASET LOADING")
print("="*80)
print(f"Original Train: {len(train_dataset_orig)} samples")
print(f"Original Val: {len(val_dataset_orig)} samples")
print(f"Merged Train: {len(train_dataset)} samples (train + val)")
print(f"Test: {len(test_dataset)} samples")
print("="*80)

# Create data loaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=2,
    pin_memory=True
)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 9. Training Function with Mixed Precision

In [ ]:
def train_model(model, train_loader, test_loader, n_epochs=N_EPOCHS):
    """Train the segmentation model with gradient accumulation and mixed precision."""
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=LEARNING_RATE * 10,
        epochs=n_epochs,
        steps_per_epoch=len(train_loader) // ACCUMULATION_STEPS,
        pct_start=0.3,
        anneal_strategy='cos'
    )
    
    scaler = GradScaler()  # For mixed precision training
    
    history = {
        'train_loss': [], 'train_iou': [], 'train_acc': [],
        'test_iou': [], 'test_acc': []
    }
    
    best_train_iou = 0.0
    
    print("\n" + "="*80)
    print("TRAINING STARTED")
    print("="*80)
    
    for epoch in range(n_epochs):
        # Training phase
        model.train()
        train_loss = train_iou = train_acc = 0.0
        optimizer.zero_grad()
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs} [Train]")
        for batch_idx, batch in enumerate(pbar):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            # Mixed precision forward pass
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, masks) / ACCUMULATION_STEPS
            
            # Backward pass with gradient accumulation
            scaler.scale(loss).backward()
            
            if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() * ACCUMULATION_STEPS
            train_iou += compute_iou(outputs, masks)
            train_acc += compute_pixel_accuracy(outputs, masks)
            
            pbar.set_postfix({
                'loss': f'{loss.item() * ACCUMULATION_STEPS:.4f}',
                'lr': f'{scheduler.get_last_lr()[0]:.6f}'
            })
        
        train_loss /= len(train_loader)
        train_iou /= len(train_loader)
        train_acc /= len(train_loader)
        
        # Test evaluation
        model.eval()
        test_iou = test_acc = 0.0
        
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Epoch {epoch+1}/{n_epochs} [Test]", leave=False):
                images = batch['image'].to(device)
                masks = batch['mask'].to(device)
                
                outputs = model(images)
                test_iou += compute_iou(outputs, masks)
                test_acc += compute_pixel_accuracy(outputs, masks)
        
        test_iou /= len(test_loader)
        test_acc /= len(test_loader)
        
        # Store history
        history['train_loss'].append(train_loss)
        history['train_iou'].append(train_iou)
        history['train_acc'].append(train_acc)
        history['test_iou'].append(test_iou)
        history['test_acc'].append(test_acc)
        
        print(f"\nEpoch {epoch+1}/{n_epochs}:")
        print(f"  Train - Loss: {train_loss:.4f}, IoU: {train_iou:.4f}, Acc: {train_acc:.4f}")
        print(f"  Test  - IoU: {test_iou:.4f}, Acc: {test_acc:.4f}")
        
        # Save checkpoints
        checkpoint_path = os.path.join(CHECKPOINTS_DIR, f"model_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_iou': train_iou,
            'test_iou': test_iou,
        }, checkpoint_path)
        
        if train_iou > best_train_iou:
            best_train_iou = train_iou
            best_path = os.path.join(CHECKPOINTS_DIR, "model_best.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'train_iou': train_iou,
                'test_iou': test_iou,
            }, best_path)
            print(f"  ★ New best! Train IoU: {train_iou:.4f}")
    
    # Save training history
    with open(os.path.join(RESULTS_DIR, "training_history.json"), 'w') as f:
        json.dump(history, f, indent=2)
    
    print("\n" + "="*80)
    print("TRAINING COMPLETE!")
    print("="*80)
    
    return history

## 10. Start Training

In [ ]:
# Train the model
history = train_model(model, train_loader, test_loader, n_epochs=N_EPOCHS)

## 11. Visualize Results

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU
axes[1].plot(history['train_iou'], label='Train IoU', linewidth=2)
axes[1].plot(history['test_iou'], label='Test IoU', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU')
axes[1].set_title('Mean IoU')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Accuracy
axes[2].plot(history['train_acc'], label='Train Accuracy', linewidth=2)
axes[2].plot(history['test_acc'], label='Test Accuracy', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Pixel Accuracy')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print final statistics
print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)
print(f"Best Train IoU: {max(history['train_iou']):.4f}")
print(f"Best Test IoU: {max(history['test_iou']):.4f}")
print(f"Final Train Accuracy: {history['train_acc'][-1]:.4f}")
print(f"Final Test Accuracy: {history['test_acc'][-1]:.4f}")
print("="*80)

## 12. Generate Predictions on Test Set

In [ ]:
# Load best model
checkpoint = torch.load(os.path.join(CHECKPOINTS_DIR, "model_best.pth"))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("Generating predictions on test set...")

# Color map for visualization
COLORS = np.array([
    [0, 0, 0],       # Background - Black
    [0, 128, 0],     # Trees - Green
    [0, 255, 0],     # Lush Bushes - Bright Green
    [255, 255, 0],   # Dry Grass - Yellow
    [139, 69, 19],   # Dry Bushes - Brown
    [160, 82, 45],   # Ground Clutter - Sienna
    [101, 67, 33],   # Logs - Dark Brown
    [128, 128, 128], # Rocks - Gray
    [210, 180, 140], # Landscape - Tan
    [135, 206, 235], # Sky - Sky Blue
], dtype=np.uint8)

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Generating predictions"):
        images = batch['image'].to(device)
        filenames = batch['filename']
        
        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1).cpu().numpy()
        
        for pred, filename in zip(predictions, filenames):
            # Save prediction as grayscale
            pred_path = os.path.join(PREDICTIONS_DIR, filename)
            cv2.imwrite(pred_path, pred.astype(np.uint8))
            
            # Save colored visualization
            colored_pred = COLORS[pred]
            colored_path = os.path.join(PREDICTIONS_DIR, f"colored_{filename}")
            cv2.imwrite(colored_path, cv2.cvtColor(colored_pred, cv2.COLOR_RGB2BGR))

print(f"✓ Predictions saved to {PREDICTIONS_DIR}")

## 13. Visualize Sample Predictions

In [ ]:
# Visualize some predictions
model.eval()
num_samples = 4

fig, axes = plt.subplots(num_samples, 3, figsize=(15, num_samples * 4))

test_dataset_raw = SegmentationDataset(
    TEST_DIR, 
    image_size=IMAGE_SIZE,
    augmentation=get_validation_augmentation()
)

with torch.no_grad():
    for i in range(num_samples):
        sample = test_dataset_raw[i * (len(test_dataset_raw) // num_samples)]
        image = sample['image'].unsqueeze(0).to(device)
        mask = sample['mask'].numpy()
        
        output = model(image)
        pred = torch.argmax(output, dim=1).cpu().numpy()[0]
        
        # Denormalize image for display
        img_display = image.cpu().numpy()[0].transpose(1, 2, 0)
        img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_display = np.clip(img_display, 0, 1)
        
        # Plot
        axes[i, 0].imshow(img_display)
        axes[i, 0].set_title('Input Image')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(COLORS[mask])
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(COLORS[pred])
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 14. Summary

### Model Specifications:
- **Resolution**: 544×960 (high-resolution for fine details)
- **Architecture**: UNet++ with EfficientNet-B5 encoder
- **Training**: Mixed precision with gradient accumulation
- **Loss**: Combined Cross Entropy + Dice Loss

### Key Features for Fine-Grained Segmentation:
1. **High Resolution**: Native 544×960 processing
2. **Multi-scale Features**: UNet++ nested skip connections
3. **Boundary Preservation**: Dice loss component
4. **Heavy Augmentation**: Robust to lighting and weather conditions
5. **Efficient Training**: Mixed precision + gradient accumulation

### Output Files:
- Checkpoints: `/kaggle/working/checkpoints/`
- Predictions: `/kaggle/working/predictions/`
- Results: `/kaggle/working/results/`